# longer runs (reruns)

**How to use this notebook:** each person opens their own copy in their own Colab session, with **Runtime > Change runtime type > T4 GPU**, and runs **one job only** (A, B or C below). A fresh Colab session forgets everything, so every job starts with its own setup cell.

| Job | What | Approx. time (from our first runs) |
|---|---|---|
| A | DiffMM, 30 epochs, one gap setting (change `RATIO`) | about 25 to 30 min |
| B | HSTGMatch with its default epochs (50 pre-train + 100 train) | about 40 min |

Keep the browser tab open while a job runs. Everything is copied to Google Drive at the end of each job.

## Job A: DiffMM, longer training

Person 1 uses `RATIO = 0.1`, person 2 uses `0.025`. If you have spare people, use `0.2` and `0.05` as well (these give the full four-point curve).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cm4604_cp3
!nvidia-smi | head -12
!git clone https://github.com/decisionintelligence/DiffMM.git
!pip install -q rtree geopandas networkx einops tqdm

In [ ]:
%%time
RATIO = 0.1      # change to 0.025, 0.2 or 0.05
EPOCHS = 30
!cd DiffMM && python main.py --city porto --keep_ratio {RATIO} --epochs {EPOCHS} --batch_size 128 --gpu_id 0 --train_flag --test_flag 2>&1 | tee ../diffmm_r{RATIO}_30ep.txt

In [ ]:
!grep -h 'test_accuracy' diffmm_r*_30ep.txt
!cp diffmm_r*_30ep.txt /content/drive/MyDrive/cm4604_cp3/

**What to look for:** `valid_accuracy` should keep rising and then flatten (the code stops early when it stops improving). Write down the final `test_accuracy` line, and the time printed by the `%%time` cell.

## Job B: HSTGMatch, default training length

Before running: upload **porto_to_hstgmatch.py** using the file panel (folder icon on the left).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cm4604_cp3
!nvidia-smi | head -12
!git clone https://github.com/decisionintelligence/DiffMM.git
!git clone https://github.com/Nerooo-g/HSTGMatch.git
!pip install -q torch_geometric pyyaml scikit-learn
!python porto_to_hstgmatch.py --porto DiffMM/data/porto --out HSTGMatch/data/porto_r0.1 --keep_ratio 0.1 --n_train 6000 --n_extra 2000

In [ ]:
%%time
# stage 1: pre-training, default 50 epochs (about 8 minutes)
!cd HSTGMatch && python scripts/pretrain.py --config configs/default.yaml --data_dir data/porto_r0.1 --batch_size 32 2>&1 | tee ../hstg_pretrain_full.txt

In [ ]:
%%time
# stage 2: supervised training, default 100 epochs (about 30 minutes)
!cd HSTGMatch && python scripts/train.py --config configs/default.yaml --data_dir data/porto_r0.1 --pretrained_encoder checkpoints/pretrain/best_encoder.pt --batch_size 32 2>&1 | tee ../hstg_train_full.txt

In [ ]:
!cp hstg_*_full.txt /content/drive/MyDrive/cm4604_cp3/
!cp -r HSTGMatch/checkpoints /content/drive/MyDrive/cm4604_cp3/hstg_checkpoints

**Decision rule for Job B:** watch the `F1=` value in the training log. If it is still below about 0.05 after roughly 30 epochs, stop the run (Runtime > Interrupt) and report honestly that HSTGMatch could not be reproduced at our data scale. Do not spend more than this one session on it.